# Планирование последовательностей интенций поверх FB — воспроизведение

Итог уже известен и получен на CPU: **метод проигрывает бейзлайну** (0.730
против 0.797, парная разность −0.067 с CI [−0.080, −0.050]). Подробности в
`REPORT.md`.

Смысл этого ноутбука — не пересчитать то же самое быстрее, а закрыть
**единственный открытый вопрос**, который CPU не потянул.

Диагноз из отчёта: узкое место — качество попарных оценок достижимости.
Корреляция стоимости с истинным расстоянием растёт с размером набора узла:

| членов в наборе | 1 | 8 | 16 | 32 |
|---|---|---|---|---|
| корреляция | 0.30 | 0.455 | 0.505 | 0.528 |

Все числа отчёта получены при **8** членах и 300 узлах — больше на CPU не
помещалось. Вопрос: если дать рёбрам лучшее качество (32 члена, 1000 узлов),
сократится ли разрыв?

Честное ожидание: скорее нет. Даже 0.528 далеко от 0.75, которые даёт прямая
оценка до цели. Но это предсказание, а не замер.

Времени займёт около полутора часов: четыре прогона по 300 эпизодов каждый.
Граф с полной конфигурацией строится один раз и кэшируется, поэтому три
последних прогона его переиспользуют.


## 1. Установка

In [ ]:
!git clone --recursive https://github.com/2FIVE192/fb-multi-intention-planning.git
%cd fb-multi-intention-planning
!pip install -q -r requirements-colab.txt

# Библиотеки графических backend'ов MuJoCo. Нужны до первого создания среды:
# какой из них заработает, выясняется пробой в следующей ячейке.
!apt-get -qq install -y libegl1 libgl1 libosmesa6 > /dev/null

In [ ]:
import os
import subprocess
import sys

# JAX по умолчанию занимает ~75% видеопамяти при первом же использовании. Если
# это сделает ядро ноутбука, дочерним процессам памяти уже не достанется.
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')


def run(*args, env=None, quiet=False):
    """Запускает скрипт репозитория отдельным процессом.

    Вывод читается построчно и печатается заново через print. Это не
    придирчивость: `subprocess.run` наследует файловые дескрипторы ядра, а в
    Colab вывод ячейки — объект Python поверх ZMQ, а не настоящий дескриптор.
    Без перепечатки весь вывод дочернего процесса уходит в лог сервера, и
    прогон выглядит немым — ровно это и случилось в прошлый раз.

    Код возврата проверяется; отрицательный означает сигнал и расшифровывается.
    """
    command = [sys.executable, '-u', *args]
    if not quiet:
        print('$', ' '.join(command[2:]), flush=True)

    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=env,
    )
    output = []
    for line in process.stdout:
        output.append(line)
        if not quiet:
            print(line, end='', flush=True)
    code = process.wait()

    if code == 0:
        return ''.join(output)

    hints = {-9: 'SIGKILL, почти всегда нехватка оперативной памяти',
             -6: 'SIGABRT, падение в нативной библиотеке (обычно CUDA или GL)',
             -11: 'SIGSEGV, падение в нативной библиотеке'}
    raise RuntimeError(f'процесс завершился с кодом {code}'
                       + (f' ({hints[code]})' if code in hints else ''))


# --- Выбор графического backend для MuJoCo ---------------------------------
# Colab работает без дисплея, а OGBench создаёт mujoco.Renderer прямо в
# MazeEnv.__init__ и сразу рендерит кадр — без рабочего контекста среда не
# поднимается вообще, даже когда рендер нам не нужен.
#
# Какой backend доступен, заранее неизвестно: egl быстрее и обычно есть на
# GPU-рантайме, но на части образов он падает по SIGSEGV прямо в драйвере.
# Поэтому не угадываем, а пробуем по очереди в ОТДЕЛЬНОМ процессе — падение
# пробы не уронит ядро. Проба идёт по тому же пути, что и настоящий код, и
# датасета не требует, поэтому стоит секунды.

PROBE = '''
import warnings; warnings.filterwarnings('ignore')
import numpy as np; np.in1d = np.isin
import sys; sys.path.insert(0, '.')
from fbplan import _upstream  # noqa: F401
from utils.env_utils import make_env_and_datasets

make_env_and_datasets('ogbench-antmaze-medium-navigate-v0', env_only=True)
print('ok')
'''


def pick_mujoco_backend():
    """Возвращает первый backend, на котором среда реально создаётся."""
    for backend in ('egl', 'osmesa', 'glfw'):
        probe_env = dict(os.environ, MUJOCO_GL=backend)
        try:
            run('-c', PROBE, env=probe_env, quiet=True)
        except RuntimeError as exc:
            print(f'  {backend:7s} не подошёл: {exc}')
            continue
        print(f'  {backend:7s} работает')
        return backend

    raise RuntimeError(
        """Ни один графический backend MuJoCo не заработал.

Обычно помогает доустановить библиотеки и перезапустить среду выполнения:
    !apt-get -qq install -y libegl1 libgl1 libosmesa6"""
    )


print('подбираю графический backend MuJoCo:')
os.environ['MUJOCO_GL'] = pick_mujoco_backend()

# GPU проверяем тоже в подпроцессе: ядро не должно трогать видеопамять вообще.
run('-c', 'import jax; print("устройства jax:", jax.devices()); '
          'assert jax.devices()[0].platform == "gpu", '
          '"GPU не подключён: Среда выполнения -> Сменить среду выполнения"')

print(f'\nMUJOCO_GL={os.environ["MUJOCO_GL"]}, окружение готово')

## 2. Данные

In [ ]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

## 3. Чекпоинты и настройки

In [ ]:
!pip -q install gdown
!python -m gdown --folder https://drive.google.com/drive/folders/1dKYhaDJH9lUREo-kUV3AwmTLrxvKO7Ek -O checkpoints

CHECKPOINT = 'checkpoints/medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isfile(os.path.join(CHECKPOINT, 'params.pkl')), 'чекпоинт не скачался'
print(sorted(os.listdir(CHECKPOINT)))

## 4. Проверки перед прогоном

Тесты логики планирования (чекпоинт не нужен, секунды) и калибровка масштабов
среды. Калибровка здесь заодно работает как первая настоящая проверка связки
«среда + датасет + чекпоинт»: если что-то не так, узнаем сейчас, а не через час.

In [ ]:
run('tests/test_planning.py')
run('scripts/calibrate.py')

## 5. Главный вопрос: помогает ли лучшее качество рёбер

Сравниваем конфигурацию из отчёта (300 узлов, 8 членов) с полной (1000 узлов,
32 члена). Всё остальное совпадает, включая отложенные сиды 1–3 — на них
подбора гиперпараметров не было.

Если разрыв с бейзлайном сократится — диагноз «дело в качестве рёбер» получает
количественное подтверждение и появляется понятное направление работы. Если
нет — значит упирается не в разрешение оценки, а в саму величину.

In [ ]:
COMMON = ['--checkpoint_dir', CHECKPOINT, '--env_name', ENV,
          '--methods', 'baseline,graph',
          '--seeds', '1,2,3', '--num_episodes', '20',
          '--replan_every', '20', '--execution', 'high',
          '--min_commit_steps', '40',
          '--tail_estimate', 'direct', '--plan_advantage_steps', '25',
          '--no_progress']

# А: ровно та конфигурация, которой получены числа отчёта (контроль).
run('scripts/run_eval.py', *COMMON,
    '--num_nodes', '300', '--num_members', '8', '--member_stride', '8',
    '--normalizer_references', '1000', '--tag', 'gpu_small')

# Б: полная конфигурация — рёбра максимального качества.
run('scripts/run_eval.py', *COMMON,
    '--num_nodes', '1000', '--num_members', '32', '--member_stride', '2',
    '--normalizer_references', '4000', '--tag', 'gpu_full')

## 6. Контрольная абляция: глубина плана

Проверка, что главный вывод отчёта воспроизводится и на хороших рёбрах.
Отличие в одном флаге: `dijkstra` — многошаговая композиция, `direct` — план из
одной подцели. На CPU было 0.47 против 0.69.

In [ ]:
for tail in ['dijkstra', 'direct']:
    run('scripts/run_eval.py', '--checkpoint_dir', CHECKPOINT, '--env_name', ENV,
        '--methods', 'graph', '--seeds', '1,2,3', '--num_episodes', '20',
        '--replan_every', '20', '--execution', 'high', '--min_commit_steps', '40',
        '--tail_estimate', tail,
        '--num_nodes', '1000', '--num_members', '32', '--member_stride', '2',
        '--normalizer_references', '4000', '--no_progress',
        '--tag', f'gpu_tail_{tail}')

## 7. Сводка

In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, '.')
from fbplan.stats import paired_comparison

TAGS = {
    'gpu_small': 'контроль: 300 узлов, 8 членов (конфигурация отчёта)',
    'gpu_full': 'полная: 1000 узлов, 32 члена',
    'gpu_tail_dijkstra': 'абляция: хвост по Дейкстре',
    'gpu_tail_direct': 'абляция: хвост одним запросом',
}

missing = []
for tag, title in TAGS.items():
    path = f'results/raw/{tag}_episodes.csv'
    if not os.path.exists(path):
        # О пропаже нужно сказать вслух: в прошлый раз сводка молча печатала
        # пустоту, и это выглядело как «результатов нет», а не «прогон упал».
        missing.append(tag)
        continue

    df = pd.read_csv(path)
    print(f'--- {title} ---')
    for method, success in df.groupby('method').success.mean().items():
        print(f'    {method:9s} {success:.3f}   ({len(df[df.method == method])} эпизодов)')

    if {'graph', 'baseline'} <= set(df.method.unique()):
        cmp = paired_comparison(df, 'graph', 'baseline')
        print(f'    парная разность {cmp["delta"]:+.3f} '
              f'[{cmp["ci_low"]:+.3f}, {cmp["ci_high"]:+.3f}] по {cmp["num_pairs"]} парам')
    print()

if missing:
    print(f'НЕТ РЕЗУЛЬТАТОВ для: {", ".join(missing)}')
    print('Соответствующие прогоны не отработали — ищите ошибку в ячейках выше.')
else:
    print('Все прогоны на месте.')
    print('\nЧто это означает. Если разрыв с бейзлайном в «полной» конфигурации '
          'заметно меньше, чем в контрольной, то диагноз отчёта — узкое место в '
          'качестве рёбер — подтверждается количественно. Если разрыв тот же, '
          'дело не в разрешении оценки, а в самой величине.')